# Top 10 cities: construction activity compared, 2017–2021

Construction activity in the 10 largest US cities over the full 5-year window, raw and normalized by city size.
Every row comes from the dbt model `top_10_cities_construction_comparison` (one row per city).

The table and charts are colored by value within each measure: **red = high, amber = middle, green = low**. Color is
scaled on a log axis because Phoenix is far above the rest on most measures; on a linear scale every
other city would look the same green.

## 1. Setup

Connection settings come from `notebooks/.env` or, per setting, the dbt profile `road_construction_autonomy`
in `~/.dbt/profiles.yml`, the same way as `weather_cohort_exploration.ipynb`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yaml
from dotenv import load_dotenv
from google.cloud import bigquery

# Notebook may be launched from the repo root or from notebooks/
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
load_dotenv(NOTEBOOK_DIR / ".env")

In [ ]:
def load_dbt_target(profile="road_construction_autonomy"):
    """Return the active target from the dbt profile, or {} if unavailable."""
    profiles_dir = Path(os.environ.get("DBT_PROFILES_DIR", Path.home() / ".dbt"))
    path = profiles_dir / "profiles.yml"
    if not path.exists():
        return {}
    prof = (yaml.safe_load(path.read_text()) or {}).get(profile, {})
    return prof.get("outputs", {}).get(prof.get("target"), {})


dbt_target = load_dbt_target()

BQ_PROJECT = os.getenv("BQ_PROJECT") or dbt_target.get("project")
BQ_LOCATION = os.getenv("BQ_LOCATION") or dbt_target.get("location", "US")
# dbt's default schema naming: <target dataset>_<custom schema> (see +schema in dbt_project.yml)
BQ_OBT_DATASET = os.getenv("BQ_OBT_DATASET") or (
    f"{dbt_target['dataset']}_obt_road_construction_and_antonomy" if dbt_target.get("dataset") else None
)

assert BQ_PROJECT and BQ_OBT_DATASET, (
    "Set BQ_PROJECT / BQ_OBT_DATASET in notebooks/.env or configure the dbt profile."
)

COMPARISON_TABLE = f"`{BQ_PROJECT}.{BQ_OBT_DATASET}.top_10_cities_construction_comparison`"
client = bigquery.Client(project=BQ_PROJECT, location=BQ_LOCATION)

## 2. Load the comparison table

In [ ]:
df = client.query(f"""
SELECT
  city,
  state,
  city_population,
  city_land_area_sq_mi,
  construction_count,
  construction_hours,
  construction_hours_per_1k_pop,
  construction_count_per_1k_pop,
  construction_hours_per_sq_mi,
  mile_hours_per_sq_mi
FROM {COMPARISON_TABLE}
""").to_dataframe()

df["city_state"] = df["city"] + ", " + df["state"]
df = df.sort_values("construction_hours", ascending=False).reset_index(drop=True)

## 3. Comparison table

Each measure column is shaded from green (lowest) through amber to red (highest); the highest value in each column is bold.
Population and land area are inputs, so they are not shaded.

In [ ]:
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm

LOW_GREEN = "#008300"
MID_AMBER = "#eda100"
HIGH_RED = "#e34948"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
LOW_TO_HIGH = LinearSegmentedColormap.from_list("low_to_high", [LOW_GREEN, MID_AMBER, HIGH_RED])

# column -> (display name, unit note used in chart titles)
MEASURES = {
    "construction_count": ("Total construction events", "Total construction events"),
    "construction_count_per_1k_pop": ("Events per 1K population", "Construction events per 1K population"),
    "construction_hours": ("Total construction hours", "Total construction hours"),
    "construction_hours_per_1k_pop": ("Hours per 1K population", "Construction hours per 1K population"),
    "construction_hours_per_sq_mi": ("Hours per sq mi", "Construction hours per sq mi"),
    "mile_hours_per_sq_mi": ("Mile-hours per sq mi", "Mile-hours (hours × miles) per sq mi"),
}


def compact(x, _pos=None):
    for div, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(x) >= div:
            return f"{x / div:.3g}{suffix}"
    return f"{x:.3g}"


def log_shades(col):
    """Background + readable text color for each cell, green (low) to red (high) on a log scale."""
    norm = LogNorm(vmin=col.min(), vmax=col.max())
    styles = []
    for v in col:
        r, g, b, _ = LOW_TO_HIGH(norm(v))
        ink = INK_PRIMARY if 0.299 * r + 0.587 * g + 0.114 * b > 0.55 else "#ffffff"
        styles.append(f"background-color: rgba({r*255:.0f},{g*255:.0f},{b*255:.0f},1); color: {ink}")
    return styles


table = df[["city_state", "city_population", "city_land_area_sq_mi", *MEASURES]].rename(
    columns={"city_state": "City", "city_population": "Population", "city_land_area_sq_mi": "Land area (sq mi)",
             **{k: v[0] for k, v in MEASURES.items()}}
)
measure_cols = [v[0] for v in MEASURES.values()]

(
    table.style
    .hide(axis="index")
    .format({"Population": "{:,.0f}", "Land area (sq mi)": "{:,.1f}", **{c: compact for c in measure_cols}})
    .apply(log_shades, subset=measure_cols)
    .highlight_max(subset=measure_cols, props="font-weight: bold")
    .set_properties(**{"text-align": "right"})
    .set_table_styles([{"selector": "th", "props": "text-align: right; font-size: 0.9em"}])
)

## 4. Charts: one measure per chart

Bars are sorted highest first and colored with the same green-to-red log scale as the table. Values are
printed on every bar, so the ranking does not depend on color alone.

In [ ]:
def plot_measure(ax, data, metric, title):
    d = data.sort_values(metric)  # barh draws bottom-up, so highest ends on top
    norm = LogNorm(vmin=d[metric].min(), vmax=d[metric].max())
    ax.barh(d["city_state"], d[metric], color=LOW_TO_HIGH(norm(d[metric])), height=0.7, edgecolor="white", linewidth=1)

    xmax = d[metric].max()
    for y, v in enumerate(d[metric]):
        ax.text(v + xmax * 0.01, y, compact(v), va="center", fontsize=9, color=INK_PRIMARY)

    ax.set_title(title, loc="left", fontsize=11.5, fontweight="bold", color=INK_PRIMARY)
    ax.set_xlim(0, xmax * 1.12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(compact))
    ax.tick_params(axis="y", labelsize=9.5, length=0, colors=INK_PRIMARY)
    ax.tick_params(axis="x", labelsize=8.5, length=0, colors=INK_SECONDARY)
    ax.grid(axis="x", color="#e6e5e0", linewidth=0.8)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color("#bdbcb6")


fig, axes = plt.subplots(3, 2, figsize=(15, 13.5), dpi=150)
for ax, (metric, (_, title)) in zip(axes.flat, MEASURES.items()):
    plot_measure(ax, df, metric, title)

fig.suptitle("Top 10 US cities: construction activity, 2017–2021", x=0.01, ha="left",
             fontsize=15, fontweight="bold")
fig.text(0.01, 0.955, "Bar color: red = highest, green = lowest within each chart (log-scaled).",
         ha="left", fontsize=10, color=INK_SECONDARY)
fig.tight_layout(rect=(0, 0, 1, 0.945), h_pad=2.5, w_pad=3)
plt.show()